# Sequence Learning Benchmark: Vanilla RNN vs LSTM
This project explores how Vanilla RNNs and LSTMs perform on two different sequence learning tasks:
1. **Time-series forecasting** : predicting household electricity consumption.
2. **Sentiment classification** : classifying IMDb movie reviews as positive or negative.

I trained and compared both models against simple baselines: a persistence model for forecasting and TF-IDF with Logistic Regression for sentiment classification.The experiments include validation-based early stopping, and the sentiment models use packed sequences to handle variable-length reviews. Run the notebook from top to bottom to download the datasets, train the models, compare their performance, and visualize the results. The default settings are intended for a quick run, with training capped at 10 epochs.
> Results can vary depending on the sample size, random seed, and runtime. All model selection is based on validation data, while the test set is reserved for final evaluation.


## 0. Setup.

In [ ]:
import random, math, re, zipfile, urllib.request, io, os, time, copy
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# Reproducible benchmark controls
FORECAST_WINDOW = 48
FORECAST_HORIZON = 1
FORECAST_BATCH_SIZE = 128
SENTIMENT_MAX_LEN = 160
SENTIMENT_BATCH_SIZE = 64
VOCAB_SIZE = 20000
TRAIN_EXAMPLES = 8000
TEST_EXAMPLES = 2000
EPOCHS = 10# train longer early stopping selects best epoch
EARLY_STOPPING_PATIENCE = 2
HIDDEN_SIZE = 64
LEARNING_RATE = 1e-3


## 1. Shared model definitions


In [ ]:
class SequenceModel(nn.Module):
    def __init__(self, kind, input_size, hidden_size, num_classes=1, embedding=None):
        super().__init__()
        self.embedding = embedding
        if embedding is not None:
            input_size = embedding.embedding_dim
        if kind.lower() == "rnn":
            self.encoder = nn.RNN(input_size, hidden_size, batch_first=True, nonlinearity="tanh")
        elif kind.lower() == "lstm":
            self.encoder = nn.LSTM(input_size, hidden_size, batch_first=True)
        else:
            raise ValueError("kind must be 'rnn' or 'lstm'")
        self.head = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths=None):
        if self.embedding is not None:
            x = self.embedding(x)
        if lengths is not None:
            packed = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            _, state = self.encoder(packed)
        else:
            _, state = self.encoder(x)
        h = state[0] if isinstance(state, tuple) else state
        return self.head(h[-1])


## 2. Task A — Electricity time-series forecasting
We use the UCI Household Electric Power Consumption dataset.

In [ ]:
UCI_URL = "https://archive.ics.uci.edu/static/public/235/individual+household+electric+power+consumption.zip"

def load_electricity():
    print("Downloading UCI household power data...")
    raw = urllib.request.urlopen(UCI_URL, timeout=60).read()
    zf = zipfile.ZipFile(io.BytesIO(raw))
    member = next(n for n in zf.namelist() if n.endswith(".txt"))
    df = pd.read_csv(zf.open(member), sep=";", na_values="?", low_memory=False)
    df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True, errors="coerce")
    df["Global_active_power"] = pd.to_numeric(df["Global_active_power"], errors="coerce")
    df = df.dropna(subset=["datetime", "Global_active_power"]).sort_values("datetime")
    series = df.set_index("datetime")["Global_active_power"].resample("30min").mean().dropna().to_numpy(dtype=np.float32)
    return series

power = load_electricity()
print("Samples:", len(power), "| min/max:", float(power.min()), float(power.max()))
plt.figure(figsize=(12,3)); plt.plot(power[:1000]); plt.title("Electricity consumption (first 1000 half-hour readings)"); plt.xlabel("Time step"); plt.ylabel("Global active power (kW)"); plt.show()


In [ ]:
# test split and train-only normalization
split = int(len(power) * 0.8)
train_raw, test_raw = power[:split], power[split:]
mean, std = train_raw.mean(), train_raw.std() + 1e-8
power_scaled = (power - mean) / std

class WindowDataset(Dataset):
    def __init__(self, values, window, horizon=1):
        self.values = torch.tensor(values, dtype=torch.float32)
        self.window, self.horizon = window, horizon
    def __len__(self):
        return len(self.values) - self.window - self.horizon + 1
    def __getitem__(self, idx):
        x = self.values[idx:idx+self.window].unsqueeze(-1)
        y = self.values[idx+self.window+self.horizon-1]
        return x, y.unsqueeze(0)
        
train_full_ds = WindowDataset(power_scaled[:split], FORECAST_WINDOW, FORECAST_HORIZON)
ts_train_end = int(len(train_full_ds) * 0.9)
train_ds = torch.utils.data.Subset(train_full_ds, range(ts_train_end))
val_ds = torch.utils.data.Subset(train_full_ds, range(ts_train_end, len(train_full_ds)))
test_ds = WindowDataset(power_scaled[split-FORECAST_WINDOW:], FORECAST_WINDOW, FORECAST_HORIZON)

train_loader_ts = DataLoader(train_ds, batch_size=FORECAST_BATCH_SIZE, shuffle=True)
val_loader_ts = DataLoader(val_ds, batch_size=FORECAST_BATCH_SIZE, shuffle=False)
test_loader_ts = DataLoader(test_ds, batch_size=FORECAST_BATCH_SIZE, shuffle=False)
print("Train windows:", len(train_ds), "| Validation windows:", len(val_ds), "| Test windows:", len(test_ds))


### Train and evaluate RNN/LSTM on forecasting


In [ ]:
def train_model(model, train_loader, loss_fn, val_loader=None, epochs=EPOCHS,
                lr=LEARNING_RATE, patience=EARLY_STOPPING_PATIENCE):
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": []}
    best_state, best_val, bad_epochs = None, float("inf"), 0

    for epoch in range(epochs):
        model.train(); total, count = 0.0, 0
        for batch in train_loader:
            if len(batch) == 3:
                xb, lengths, yb = batch
                lengths = lengths.to("cpu")
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb, lengths=lengths)
            else:
                xb, yb = batch
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                pred = model(xb)
            optimizer.zero_grad()
            loss = loss_fn(pred, yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            total += loss.item() * len(xb); count += len(xb)
        train_loss = total / max(count, 1)
        history["train_loss"].append(train_loss)

        if val_loader is not None:
            model.eval(); val_total, val_count = 0.0, 0
            with torch.no_grad():
                for batch in val_loader:
                    if len(batch) == 3:
                        xb, lengths, yb = batch
                        pred = model(xb.to(DEVICE), lengths=lengths)
                    else:
                        xb, yb = batch
                        pred = model(xb.to(DEVICE))
                    yb = yb.to(DEVICE)
                    val_total += loss_fn(pred, yb).item() * len(xb)
                    val_count += len(xb)
            val_loss = val_total / max(val_count, 1)
            history["val_loss"].append(val_loss)
            print(f"Epoch {epoch+1}/{epochs} — train: {train_loss:.4f} | val: {val_loss:.4f}")
            if val_loss < best_val:
                best_val = val_loss
                best_state = copy.deepcopy(model.state_dict())
                bad_epochs = 0
            else:
                bad_epochs += 1
                if bad_epochs >= patience:
                    print(f"Early stopping at epoch {epoch+1}; best validation loss: {best_val:.4f}")
                    break
        else:
            print(f"Epoch {epoch+1}/{epochs} — train loss: {train_loss:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    return history

@torch.no_grad()
def evaluate_forecasting(model, loader):
    model.eval()
    preds, targets = [], []
    for xb, yb in loader:
        preds.append(model(xb.to(DEVICE)).cpu().numpy())
        targets.append(yb.numpy())
    p = np.concatenate(preds).reshape(-1) * std + mean
    y = np.concatenate(targets).reshape(-1) * std + mean
    rmse = float(np.sqrt(np.mean((p-y)**2)))
    mae = float(np.mean(np.abs(p-y)))
    return rmse, mae, p, y

forecast_results = {}
for kind in ["RNN", "LSTM"]:
    print("\\n###", kind, "forecasting")
    model = SequenceModel(kind, input_size=1, hidden_size=HIDDEN_SIZE, num_classes=1)
    hist = train_model(model, train_loader_ts, nn.MSELoss(), val_loader=val_loader_ts)
    rmse, mae, pred, actual = evaluate_forecasting(model, test_loader_ts)
    forecast_results[kind] = {"history": hist, "rmse": rmse, "mae": mae, "pred": pred, "actual": actual}
    print(f"Test RMSE: {rmse:.4f} kW | MAE: {mae:.4f} kW")

#  forecast the next point using the most recent observed point
persist_scaled = np.array([test_ds[i][0][-1, 0].item() for i in range(len(test_ds))])
persist_pred = persist_scaled * std + mean
persist_actual = np.array([test_ds[i][1][0].item() for i in range(len(test_ds))]) * std + mean
persistence_rmse = float(np.sqrt(np.mean((persist_pred - persist_actual)**2)))
persistence_mae = float(np.mean(np.abs(persist_pred - persist_actual)))
forecast_results["Persistence"] = {
    "rmse": persistence_rmse, "mae": persistence_mae,
    "pred": persist_pred, "actual": persist_actual
}
print(f"Persistence baseline — RMSE: {persistence_rmse:.4f} kW | MAE: {persistence_mae:.4f} kW")

plt.figure(figsize=(10,4))
for kind, result in forecast_results.items():
    if "history" in result:
        plt.plot(result["history"]["train_loss"], marker="o", label=f"{kind} train")
        plt.plot(result["history"]["val_loss"], marker=".", linestyle="--", label=f"{kind} val")
plt.title("Forecasting train/validation loss"); plt.xlabel("Epoch"); plt.ylabel("MSE"); plt.legend(); plt.show()

plt.figure(figsize=(12,4))
actual = forecast_results["RNN"]["actual"]
plt.plot(actual[:300], label="Actual", linewidth=2)
for kind, result in forecast_results.items():
    plt.plot(result["pred"][:300], label=f"{kind} prediction", alpha=.8)
plt.title("Electricity forecasting: actual vs predictions (first 300 test points)")
plt.xlabel("Test time step"); plt.ylabel("Power (kW)"); plt.legend(); plt.show()

pd.DataFrame([{ "Model": k, "RMSE (kW)": v["rmse"], "MAE (kW)": v["mae"] } for k,v in forecast_results.items()])


## 3. Task B — IMDb sentiment classification


In [ ]:
from datasets import load_dataset

print("Loading IMDb dataset...")
imdb = load_dataset("stanfordnlp/imdb")

# sample from the original train/test splits
train_rows_all = imdb["train"].shuffle(seed=SEED).select(
    range(min(TRAIN_EXAMPLES, len(imdb["train"])))
)
test_rows = imdb["test"].shuffle(seed=SEED + 1).select(
    range(min(TEST_EXAMPLES, len(imdb["test"])))
)

#  validation slice from the sampled training examples
val_count = max(1, int(0.1 * len(train_rows_all)))
train_rows = train_rows_all.select(range(len(train_rows_all) - val_count))
val_rows = train_rows_all.select(range(len(train_rows_all) - val_count, len(train_rows_all)))

print("Train:", len(train_rows), "| Validation:", len(val_rows), "| Test:", len(test_rows))


In [ ]:
TOKEN_RE = re.compile(r"[a-z]+|[0-9]+", re.IGNORECASE)
def tokenize(text):
    return TOKEN_RE.findall(text.lower())

counter = Counter()
for row in train_rows:
    counter.update(tokenize(row["text"]))
vocab = {"<PAD>": 0, "<UNK>": 1}
for token, _ in counter.most_common(VOCAB_SIZE - 2):
    vocab[token] = len(vocab)
print("Vocabulary size:", len(vocab))

def encode_text_with_length(text, max_len=SENTIMENT_MAX_LEN):
    ids = [vocab.get(tok, 1) for tok in tokenize(text)[:max_len]]
    length = max(1, len(ids))  # pack_padded_sequence requires positive lengths
    ids = ids + [0] * (max_len - len(ids))
    return torch.tensor(ids, dtype=torch.long), length

def encode_text(text, max_len=SENTIMENT_MAX_LEN):
    return encode_text_with_length(text, max_len)[0]

class IMDbDataset(Dataset):
    def __init__(self, rows):
        self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        row = self.rows[idx]
        ids, length = encode_text_with_length(row["text"])
        label = torch.tensor([float(row["label"])], dtype=torch.float32)
        return ids, torch.tensor(length, dtype=torch.long), label

train_ds_sent = IMDbDataset(train_rows)
val_ds_sent = IMDbDataset(val_rows)
test_ds_sent = IMDbDataset(test_rows)
train_loader_sent = DataLoader(train_ds_sent, batch_size=SENTIMENT_BATCH_SIZE, shuffle=True)
val_loader_sent = DataLoader(val_ds_sent, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False)
test_loader_sent = DataLoader(test_ds_sent, batch_size=SENTIMENT_BATCH_SIZE, shuffle=False)


### Train and evaluate RNN/LSTM on sentiment


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

@torch.no_grad()
def evaluate_sentiment(model, loader):
    model.eval()
    all_probs, all_y = [], []
    for xb, lengths, yb in loader:
        logits = model(xb.to(DEVICE), lengths=lengths)
        probs = torch.sigmoid(logits).cpu().numpy().reshape(-1)
        all_probs.extend(probs)
        all_y.extend(yb.numpy().reshape(-1))
    pred = (np.array(all_probs) >= 0.5).astype(int)
    y = np.array(all_y).astype(int)
    return {
        "accuracy": accuracy_score(y, pred),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "y": y, "pred": pred
    }

sentiment_results = {}
for kind in ["RNN", "LSTM"]:
    print("\\n###", kind, "sentiment")
    emb = nn.Embedding(len(vocab), 64, padding_idx=0)
    model = SequenceModel(kind, input_size=64, hidden_size=HIDDEN_SIZE, num_classes=1, embedding=emb)
    hist = train_model(model, train_loader_sent, nn.BCEWithLogitsLoss(), val_loader=val_loader_sent)
    metrics = evaluate_sentiment(model, test_loader_sent)
    sentiment_results[kind] = {"history": hist, **metrics}
    print({m: round(metrics[m], 4) for m in ["accuracy", "precision", "recall", "f1"]})

# TF-IDF features + Logistic Regression, evaluated on the same sampled test set.
print("\\n### TF-IDF + Logistic Regression baseline")
tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, stop_words="english")
X_train_tfidf = tfidf.fit_transform([row["text"] for row in train_rows])
y_train_tfidf = np.array([row["label"] for row in train_rows])
X_test_tfidf = tfidf.transform([row["text"] for row in test_rows])
y_test_tfidf = np.array([row["label"] for row in test_rows])
tfidf_lr = LogisticRegression(max_iter=1000, C=2.0)
tfidf_lr.fit(X_train_tfidf, y_train_tfidf)
tfidf_pred = tfidf_lr.predict(X_test_tfidf)
tfidf_metrics = {
    "accuracy": accuracy_score(y_test_tfidf, tfidf_pred),
    "precision": precision_score(y_test_tfidf, tfidf_pred, zero_division=0),
    "recall": recall_score(y_test_tfidf, tfidf_pred, zero_division=0),
    "f1": f1_score(y_test_tfidf, tfidf_pred, zero_division=0),
    "y": y_test_tfidf, "pred": tfidf_pred
}
sentiment_results["TF-IDF + Logistic Regression"] = tfidf_metrics
print({m: round(tfidf_metrics[m], 4) for m in ["accuracy", "precision", "recall", "f1"]})

plt.figure(figsize=(8,4))
for kind, result in sentiment_results.items():
    if "history" in result:
        plt.plot(result["history"]["train_loss"], marker="o", label=f"{kind} train")
        plt.plot(result["history"]["val_loss"], marker=".", linestyle="--", label=f"{kind} val")
plt.title("Sentiment train/validation loss"); plt.xlabel("Epoch"); plt.ylabel("BCE loss"); plt.legend(); plt.show()

fig, axes = plt.subplots(1, 3, figsize=(13,4))
for ax, kind in zip(axes, ["RNN", "LSTM", "TF-IDF + Logistic Regression"]):
    ConfusionMatrixDisplay(confusion_matrix(sentiment_results[kind]["y"], sentiment_results[kind]["pred"]),
                          display_labels=["Negative", "Positive"]).plot(ax=ax, colorbar=False)
    ax.set_title(kind)
plt.tight_layout(); plt.show()

pd.DataFrame([{ "Model": k, **{m: v[m] for m in ["accuracy", "precision", "recall", "f1"]} } for k,v in sentiment_results.items()])


## 4. Compare and summarize


In [ ]:
summary = []
for kind, res in forecast_results.items():
    summary.append({"Task": "Forecasting", "Model": kind, "RMSE": res["rmse"], "MAE": res["mae"],
                    "Accuracy": np.nan, "F1": np.nan})
for kind, res in sentiment_results.items():
    summary.append({"Task": "Sentiment", "Model": kind, "RMSE": np.nan, "MAE": np.nan,
                    "Accuracy": res["accuracy"], "F1": res["f1"]})
summary_df = pd.DataFrame(summary)
display(summary_df)
